# Pandas Reshaping Mastery
## pivot → melt → stack → unstack → explode → crosstab
## The Final Shape-Shifting Weapon

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load our final cleaned master dataset
df = pd.read_csv('master_ecommerce_2025.csv', parse_dates=['order_date'])
print(f"Final dataset ready: {df.shape}")
df.head()

### Exercise 1: Classic pivot_table – revenue by country & category

In [ ]:
pivot = df.pivot_table(
    values='total_amount',
    index='country',
    columns='category',
    aggfunc='sum',
    fill_value=0,
    margins=True
)

pivot.style.format('{:,.0f}').background_gradient(cmap='Blues')

### Exercise 2: Multi-level pivot – month + country

In [ ]:
df['month'] = df['order_date'].dt.strftime('%b')

monthly = df.pivot_table(
    values='total_amount',
    index='country',
    columns='month',
    aggfunc='sum',
    fill_value=0
).round(0)

plt.figure(figsize=(12,8))
sns.heatmap(monthly, annot=True, fmt=',.0f', cmap='YlOrRd')
plt.title('Revenue Heatmap: Country × Month')
plt.show()

### Exercise 3: melt() – wide to long (for seaborn/plotly)

In [ ]:
wide = monthly.reset_index()
long = pd.melt(wide, id_vars='country', var_name='month', value_name='revenue')

plt.figure(figsize=(12,6))
sns.barplot(data=long, x='country', y='revenue', hue='month')
plt.title('Revenue by Country & Month – Long Format (melted)')
plt.xticks(rotation=45)
plt.show()

### Exercise 4: stack() & unstack() – MultiIndex magic

In [ ]:
stacked = pivot.stack()
print("Stacked (long format):")
print(stacked.head(10))

unstacked = stacked.unstack()
print("\nBack to wide with unstack():")
print(unstacked.equals(pivot))  # True!

### Exercise 5: explode() – handle list columns (real JSON data)

In [ ]:
# Simulate orders with multiple products in one cell
df_lists = pd.DataFrame({
    'order_id': ['A1', 'A2', 'A3'],
    'products': [ ['iPhone', 'Case'], ['Laptop'], ['Shoes', 'Socks', 'Hat'] ],
    'revenue': [1500, 2400, 380]
})

exploded = df_lists.explode('products')
print("One row per product:")
exploded

### Exercise 6: crosstab – quick frequency tables

In [ ]:
pd.crosstab(df['country'], df['payment_method'], normalize='index').round(3).style.background_gradient(cmap='Greens')

### Exercise 7: pivot with multiple values

In [ ]:
multi = df.pivot_table(
    values=['total_amount', 'quantity'],
    index='country',
    columns='is_weekend',
    aggfunc={'total_amount': 'sum', 'quantity': 'mean'}
)
multi

### Exercise 8: FINAL BOSS – Reshape for Machine Learning

In [ ]:
def prepare_for_ml(df):
    # 1. Customer-level aggregation
    customer_features = df.groupby('customer_id').agg({
        'total_amount': ['sum', 'mean', 'count'],
        'order_date': ['min', 'max'],
        'category': lambda x: x.nunique(),
        'country': 'first',
        'returned': 'mean'
